# Transmon 큐비트 기초

이 노트북에서는 `ScQubitsMimic.jl`을 사용하여 Transmon 큐비트의 기본적인 물리량을 계산합니다.

**다루는 내용:**
- 에너지 고유값과 전이 주파수
- 비조화성 (anharmonicity)
- 전하 기저 파동함수
- 전하 분산 (charge dispersion)
- $E_J$ 파라미터 스윕
- 행렬 요소
- 회로 기반 검증

In [9]:
using ScQubitsMimic
using CairoMakie

ArgumentError: ArgumentError: Package CairoMakie not found in current path.
- Run `import Pkg; Pkg.add("CairoMakie")` to install the CairoMakie package.

## 1. Transmon 해밀토니안

Transmon 큐비트의 해밀토니안은 쿠퍼 쌍 수 연산자 $\hat{n}$과 위상 연산자 $\hat{\varphi}$로 표현됩니다:

$$H = 4E_C(\hat{n} - n_g)^2 - E_J\cos\hat{\varphi}$$

여기서:
- $E_C$: 충전 에너지 (charging energy)
- $E_J$: 조셉슨 에너지 (Josephson energy)
- $n_g$: 오프셋 전하 (게이트 전하)

Transmon 영역은 $E_J/E_C \gg 1$인 경우에 해당하며, 이 조건에서 전하 잡음에 대한 감도가 지수적으로 억제됩니다.

In [10]:
# Transmon 큐비트 생성
tmon = Transmon(EJ=30.0, EC=1.2, ng=0.0, ncut=30, truncated_dim=6)

println("Transmon 파라미터:")
println("  EJ = $(tmon.EJ) GHz")
println("  EC = $(tmon.EC) GHz")
println("  EJ/EC = $(tmon.EJ / tmon.EC)")
println("  ncut = $(tmon.ncut)")
println("  힐베르트 공간 차원: $(hilbertdim(tmon))")

Transmon 파라미터:
  EJ = 30.0 GHz
  EC = 1.2 GHz
  EJ/EC = 25.0
  ncut = 30
  힐베르트 공간 차원: 61


## 2. 에너지 고유값

전하 기저 $|n\rangle$ ($n = -n_{\text{cut}}, \ldots, n_{\text{cut}}$)에서 해밀토니안을 대각화하여 에너지 고유값을 구합니다.

In [11]:
evals = eigenvals(tmon; evals_count=6)

println("처음 6개 에너지 고유값 (GHz):")
for (i, e) in enumerate(evals)
    println("  E_$(i-1) = $(round(e, digits=6))")
end

println("\n바닥 상태 기준 전이 주파수:")
for i in 2:6
    println("  ω_{0→$(i-1)} = $(round(evals[i] - evals[1], digits=6)) GHz")
end

처음 6개 에너지 고유값 (GHz):
  E_0 = -21.826741
  E_1 = -6.159599
  E_2 = 7.941246
  E_3 = 20.840802
  E_4 = 28.098985
  E_5 = 45.795058

바닥 상태 기준 전이 주파수:
  ω_{0→1} = 15.667142 GHz
  ω_{0→2} = 29.767987 GHz
  ω_{0→3} = 42.667543 GHz
  ω_{0→4} = 49.925726 GHz
  ω_{0→5} = 67.621799 GHz


## 3. 비조화성 (Anharmonicity)

Transmon 큐비트가 조화 진동자와 구별되는 핵심 특성은 **비조화성**입니다:

$$\alpha = \omega_{12} - \omega_{01}$$

Transmon 영역에서 비조화성은 충전 에너지에 의해 결정됩니다:

$$\alpha \approx -E_C$$

또한, 기본 전이 주파수의 근사식은 다음과 같습니다:

$$\omega_{01} \approx \sqrt{8E_JE_C} - E_C$$

In [12]:
ω01 = evals[2] - evals[1]
ω12 = evals[3] - evals[2]
α = ω12 - ω01

println("ω₀₁ = $(round(ω01, digits=6)) GHz")
println("ω₁₂ = $(round(ω12, digits=6)) GHz")
println("비조화성 α = $(round(α, digits=6)) GHz")
println("기대값 (transmon 극한): α ≈ -EC = $(round(-tmon.EC, digits=6)) GHz")

# Transmon 근사 검증
ω_approx = sqrt(8 * tmon.EJ * tmon.EC) - tmon.EC
println("\nTransmon 근사: ω₀₁ ≈ √(8 EJ EC) - EC = $(round(ω_approx, digits=6)) GHz")
println("정확한 값: ω₀₁ = $(round(ω01, digits=6)) GHz")
println("오차: $(round(abs(ω01 - ω_approx) / ω01 * 100, digits=3))%")

ω₀₁ = 15.667142 GHz
ω₁₂ = 14.100844 GHz
비조화성 α = -1.566298 GHz
기대값 (transmon 극한): α ≈ -EC = -1.2 GHz

Transmon 근사: ω₀₁ ≈ √(8 EJ EC) - EC = 15.770563 GHz
정확한 값: ω₀₁ = 15.667142 GHz
오차: 0.66%


## 4. 전하 기저 파동함수

Transmon의 고유 상태를 전하 기저 $|n\rangle$에서 시각화합니다. Transmon 영역($E_J/E_C \gg 1$)에서 파동함수는 $n=0$ 부근에 집중되며, 높은 에너지 준위일수록 더 넓게 퍼집니다.

In [13]:
plot_wavefunction(tmon, [1, 2, 3, 4]; mode=:abs_sqr)

MethodError: MethodError: no method matching plot_wavefunction(::Transmon, ::Vector{Int64}; mode::Symbol)
The function `plot_wavefunction` exists, but no method is defined for this combination of argument types.

## 5. 전하 분산 (Charge Dispersion)

오프셋 전하 $n_g$를 변화시키면 전이 주파수가 달라집니다. Transmon 영역에서 이 전하 분산은 지수적으로 작아집니다:

$$\epsilon_m \propto e^{-\sqrt{8E_J/E_C}}$$

이것이 Transmon이 전하 잡음에 강한 이유입니다.

In [14]:
println("전하 분산 (ω₀₁ vs 오프셋 전하 ng):")
ng_vals = [0.0, 0.1, 0.2, 0.25, 0.3, 0.4, 0.5]
w01_vals = Float64[]
for ng in ng_vals
    tmon.ng = ng
    e = eigenvals(tmon; evals_count=2)
    w01 = e[2] - e[1]
    push!(w01_vals, w01)
    println("  ng = $ng → ω₀₁ = $(round(w01, digits=8)) GHz")
end
tmon.ng = 0.0  # 초기화

# 전하 분산 크기
dispersion = maximum(w01_vals) - minimum(w01_vals)
println("\n전하 분산 크기: $(round(dispersion * 1e6, digits=2)) kHz")

전하 분산 (ω₀₁ vs 오프셋 전하 ng):
  ng = 0.0 → ω₀₁ = 15.66714237 GHz
  ng = 0.1 → ω₀₁ = 15.66652694 GHz
  ng = 0.2 → ω₀₁ = 15.66491651 GHz
  ng = 0.25 → ω₀₁ = 15.66392177 GHz
  ng = 0.3 → ω₀₁ = 15.66292746 GHz
  ng = 0.4 → ω₀₁ = 15.66131952 GHz
  ng = 0.5 → ω₀₁ = 15.66070564 GHz

전하 분산 크기: 6436.72 kHz


## 6. $E_J$ 파라미터 스윕

조셉슨 에너지 $E_J$를 변화시키면서 에너지 스펙트럼의 변화를 관찰합니다. $E_J$가 증가할수록 전이 주파수가 높아지고, $E_J/E_C$ 비가 커지면서 비조화성의 상대적 크기가 감소합니다.

In [15]:
sweep = ParameterSweep(tmon, :EJ, range(10.0, 50.0, length=50); evals_count=6)
plot_evals_vs_paramvals(sweep; subtract_ground=true, evals_count=6)

MethodError: MethodError: no method matching plot_evals_vs_paramvals(::ParameterSweep; subtract_ground::Bool, evals_count::Int64)
The function `plot_evals_vs_paramvals` exists, but no method is defined for this combination of argument types.

## 7. 행렬 요소

전하 연산자 $\hat{n}$의 행렬 요소 $\langle i|\hat{n}|j\rangle$는 큐비트와 외부 시스템 간의 결합 강도를 결정합니다. 쌍극자 결합 (capacitive coupling)에서는 전하 연산자가 결합 연산자 역할을 합니다.

선택 규칙에 의해 $|\Delta m| = 1$인 전이가 지배적입니다.

In [16]:
n_op = n_operator(tmon)
mel_table = matrixelement_table(tmon, n_op; evals_count=6)

println("전하 연산자 행렬 요소 ⟨i|n̂|j⟩:")
for i in 1:6, j in 1:6
    v = mel_table[i, j]
    if abs(v) > 1e-6
        println("  ⟨$(i-1)|n̂|$(j-1)⟩ = $(round(v, digits=6))")
    end
end

전하 연산자 행렬 요소 ⟨i|n̂|j⟩:
  ⟨0|n̂|1⟩ = -0.902685 + 0.0im
  ⟨0|n̂|3⟩ = -0.050038 + 0.0im
  ⟨0|n̂|5⟩ = 0.005441 + 0.0im
  ⟨1|n̂|0⟩ = -0.902685 + 0.0im
  ⟨1|n̂|2⟩ = -1.212436 + 0.0im
  ⟨1|n̂|4⟩ = -0.13048 + 0.0im
  ⟨2|n̂|1⟩ = -1.212436 + 0.0im
  ⟨2|n̂|3⟩ = -1.358579 + 0.0im
  ⟨2|n̂|5⟩ = 0.158931 + 0.0im
  ⟨3|n̂|0⟩ = -0.050038 + 0.0im
  ⟨3|n̂|2⟩ = -1.358579 + 0.0im
  ⟨3|n̂|4⟩ = -1.564707 + 0.0im
  ⟨4|n̂|1⟩ = -0.13048 + 0.0im
  ⟨4|n̂|3⟩ = -1.564707 + 0.0im
  ⟨4|n̂|5⟩ = 0.808977 + 0.0im
  ⟨5|n̂|0⟩ = 0.005441 + 0.0im
  ⟨5|n̂|2⟩ = 0.158931 + 0.0im
  ⟨5|n̂|4⟩ = 0.808977 + 0.0im


In [17]:
plot_matrixelements(tmon, s -> n_operator(s); evals_count=6, mode=:abs)

MethodError: MethodError: no method matching plot_matrixelements(::Transmon, ::var"#23#24"; evals_count::Int64, mode::Symbol)
The function `plot_matrixelements` exists, but no method is defined for this combination of argument types.

## 8. 회로 기반 검증

`Circuit`을 사용하여 YAML 기술에서 직접 Transmon을 구성하고, 하드코딩된 `Transmon` 타입과 비교하여 검증합니다.

단일 JJ 브랜치의 EC 파라미터와 Transmon의 총 $E_C$의 관계:

$$E_{C,\text{total}} = \frac{e^2}{2C} = 4 \times E_{C,\text{branch}}$$

따라서 $E_C = 1.2$ GHz를 얻으려면 $E_{C,\text{branch}} = 0.3$ GHz로 설정해야 합니다.

In [18]:
desc = """
branches:
  - [JJ, 0, 1, EJ=30.0, EC=0.3]
"""
circ = Circuit(desc; ncut=30)

evals_circ = eigenvals(circ; evals_count=6)
evals_hard = eigenvals(Transmon(EJ=30.0, EC=1.2, ncut=30, truncated_dim=6); evals_count=6)

println("  준위    Circuit         Transmon        차이")
println("  " * "-"^55)
for i in 1:6
    diff = abs(evals_circ[i] - evals_hard[i])
    println("  E_$(i-1)     $(round(evals_circ[i], digits=8))    $(round(evals_hard[i], digits=8))    $(diff)")
end
println("\n  최대 차이: $(maximum(abs.(evals_circ .- evals_hard)))")
println("  검증: ", maximum(abs.(evals_circ .- evals_hard)) < 1e-10 ? "통과 ✓" : "실패 ✗")

  준위    Circuit         Transmon        차이
  -------------------------------------------------------
  E_0     -21.82674091    -21.82674091    0.0
  E_1     -6.15959855    -6.15959855    0.0
  E_2     7.94124586    7.94124586    0.0
  E_3     20.84080224    20.84080224    0.0
  E_4     28.09898516    28.09898516    0.0
  E_5     45.79505843    45.79505843    0.0

  최대 차이: 0.0
  검증: 통과 ✓
